# Libya land-cover mapping assistant — WP06

دستیار اولیهٔ گفت‌وگو با موتور موجود. کد مرجع GEE و روش مدل تغییری نمی‌کنند.

**برای اولین تجربه، مدل را دوباره اجرا نکن:** سلول‌های ۱ تا ۴ را اجرا کن، ZIP پایلوت موفق (`wp05-pilot-…-7.zip`) را در سلول ۳ وارد کن و در چت بنویس `نتیجه` یا `وضعیت`.

حالت `guided` فرمان‌های مشخص را می‌فهمد و کلید API لازم ندارد. حالت `openai` درخواست آزاد فارسی/انگلیسی را با کلید API می‌فهمد. تنظیمات هر اجرای تازه ابتدا نمایش داده می‌شود؛ فقط دکمهٔ **تأیید و اجرا / Run** کار را شروع می‌کند.

خروجی پایلوت فقط یک قطعه است. خروجی کامل منطقه، آمار مساحت کلاس‌ها و تحلیل تغییرات در این مرحله آماده نیستند. وضعیت واردشده از ZIP با زمان ثبت آن نشان داده می‌شود.


## 1. نصب نسخهٔ ثابت — در یک Runtime تازه

In [ ]:
import sys, subprocess
SERVICE_COMMIT = "e77c230d7153de5c741bd29e46dc35f15b0130ab"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
    "libya-landcover-assistant[chat] @ git+https://github.com/hamedsabzchi/libya-automated-land-cover-mapping.git@" + SERVICE_COMMIT])
import ee, libya_landcover
if libya_landcover.__version__ != "0.6.0":
    raise RuntimeError("Start a fresh runtime: an older package is already imported")
assert ee.__version__ == "1.7.48"
print("Assistant:", libya_landcover.__version__, SERVICE_COMMIT)


## 2. انتخاب حالت و پروفایل

برای گفت‌وگوی آزاد `MODE = "openai"` را انتخاب کن. کلید را فقط در ورودی مخفی وارد کن؛ آن را در سلول کد یا چت ننویس. در حالت OpenAI پیام تو، تنظیمات فعال/پیشنهادی و حداکثر یک سؤال روشن‌کنندهٔ قبلی به API ارسال می‌شود؛ گراف‌ها و داده‌های مدل ارسال نمی‌شوند. مدل نمونه قابل تغییر است و باید در حساب API تو در دسترس باشد.

پروفایل پیش‌فرض همان طرابلس و Training Asset قبلی تو است. سلول ۲ را دوباره اجرا کنی، پیش‌نمایش تأیید قبلی دیگر فعال نیست. رکوردهای کار در همین Runtime باقی می‌مانند.


In [ ]:
from pathlib import Path
from getpass import getpass
from libya_landcover.assistant import Assistant, TRIPOLI
from libya_landcover.chat_language import GuidedParser, OpenAIParser
from libya_landcover.job_store import JobStore

MODE = "guided"  # @param ["guided", "openai"]
OPENAI_MODEL = "gpt-4.1-mini"  # @param {type:"string"}
PROFILE = dict(TRIPOLI)
STORE_DIR = Path("/content/assistant-store")
store = JobStore(STORE_DIR)
if MODE == "openai":
    parser = OpenAIParser(getpass("OpenAI API key (hidden): "), OPENAI_MODEL)
elif MODE == "guided":
    parser = GuidedParser()
else:
    raise ValueError("MODE must be guided or openai")
assistant = Assistant(store, engine_commit=SERVICE_COMMIT, profile=PROFILE, parser=parser)
print(parser.label)
print("Active project:", PROFILE["project"])
print("AOI:", PROFILE["aoi_asset"])
print("Training:", PROFILE["training_asset"])
print("Target year:", PROFILE["target_year"])


## 3. ورود پشتیبان موجود — اختیاری، بدون اجرای مدل یا تماس با GEE

ZIP سلول ۷ پایلوت موفق یا ZIP اجرای کامل‌شدهٔ WP04 را انتخاب کن. برای شروع یک مدل تازه می‌توانی این سلول را رد کنی. برای ورود چند فایل، هر بار همین سلول را با یک ZIP اجرا کن. نام فایل ورودی مسیر استخراج نمی‌شود. بعد از ورود، سلول ۴ را اجرا کن.


In [ ]:
from google.colab import files
if "assistant" not in globals():
    raise RuntimeError("Run cells 1 and 2 first")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose exactly one completed mapping or pilot ZIP")
archive_path = STORE_DIR / "assistant-upload.zip"
archive_path.write_bytes(next(iter(uploaded.values())))
try:
    imported = assistant.import_archive(archive_path)
finally:
    archive_path.unlink(missing_ok=True)
    uploaded.clear()
print("Imported:", imported["job_id"], imported["state"])
print("Source:", imported["source"])
print("No model or export was started. Open chat with cell 4.")


## 4. بازکردن چت

در حالت راهنما: `نتیجه`، `وضعیت`، `jobs`، `map 2024`.

در حالت OpenAI: «نتیجهٔ کار انتخاب‌شده چیست؟» یا «برای طرابلس نقشهٔ سال ۲۰۲۴ را آماده کن». برای اجرای تازه یا Refresh زنده، ابتدا سلول ۵ را اجرا کن و به همین رابط برگرد.

دکمهٔ **Download report** گزارش وضعیت را می‌دهد. **Download backup** پشتیبان قابل ورود مجددِ نقشهٔ کامل‌شده/پایلوت را می‌دهد. TIFF را از لینک Drive بگیر؛ در ZIP پشتیبان پیکسل‌های تصویر نیست.


In [ ]:
from google.colab import output, files
from libya_landcover.chat_ui import display_chat
if "assistant" not in globals():
    raise RuntimeError("Run cells 1 and 2 first")
output.enable_custom_widget_manager()
ui = display_chat(assistant, download=files.download)


## 5. ورود GEE — فقط پیش از عملیات زنده

برای خواندن ZIP و گزارش قبلی لازم نیست. برای Refresh سرور، تهیهٔ پایلوت یا اجرای مدل تازه این سلول را اجرا کن؛ سپس از چت بالا استفاده کن. ورود به حساب همان دسترسی‌های قبلی پروژه و Assetها را می‌خواهد. این سلول هیچ مدل یا Export جدیدی شروع نمی‌کند.


In [ ]:
import ee
if "assistant" not in globals():
    raise RuntimeError("Run cells 1 and 2 first")
ee.Authenticate()
ee.Initialize(project=PROFILE["project"])
print("Earth Engine initialized. Return to chat; no model/export was started.")


### پس از قطع Runtime

سلول‌های ۱ تا ۴ را اجرا و آخرین ZIP پشتیبان را دوباره وارد کن. برای به‌روزکردن وضعیت Export، سلول ۵ و سپس **Refresh status** را بزن. پیام یا دکمهٔ اجرا را برای صرفاً دیدن وضعیت تکرار نکن.

اجرای Python مدل داخل Runtime است؛ هنگام اجرای تازه، Runtime را متصل نگه دار. این نسخه پشتیبان اجرای نیمه‌تمام مدل را بازیابی نمی‌کند. Export ثبت‌شدهٔ GEE مستقل از Runtime است و از طریق پشتیبان پایلوت قابل پیگیری است. وضعیت درخواست بزرگ قبلی از این نوت‌بوک تغییر نمی‌کند.

**محدودهٔ آزمون:** منطق و رابط با آزمون‌های آفلاین و پاسخ‌های شبیه‌سازی‌شده بررسی شده‌اند؛ کیفیت گفت‌وگوی زندهٔ OpenAI و اجرای GEE از این رابط هنوز باید در حساب خودت امتحان شوند. برای تجربهٔ اول، همان پایلوت موفق کافی است.
